# Clean NNDSS influenza data and get state level aggregates; merge with population dataset


Author: Aminath Shausan

This notebook cleans and aggregates NNDSS influenza notifications by states and territories, perform interpolation for the 5 missing weeks for NT, and and merge with population dataset

NNDSS data was gathered from: https://www.health.gov.au/resources/publications/national-notifiable-diseases-surveillance-system-nndss-public-dataset-influenza-laboratory-confirmed?language=en

Data is from 01 Jan 2008 to 31 Dec 2022.  ACT is not included in the public dataset



#### **Load libraries**

In [ ]:
path = '<PUT PATH TO FOLDER>'
import warnings
warnings.filterwarnings("ignore")


In [ ]:
# #import libraries
from datetime import datetime
import pandas as pd
import numpy as np
from pandas import read_excel    
import itertools

## Format influenza data 


In [ ]:
#load raw data and sort by 'week' column
sheet1 = '2008-2015' #  
sheet2 = '2016-2018' 
sheet3 = '2019-2022' 
file_name = path + 'data/pathogen/nndss_influenza_raw_data.xlsx' # 
df1 = read_excel(file_name, sheet_name = sheet1)
df2 = read_excel(file_name, sheet_name = sheet2)
df3 = read_excel(file_name, sheet_name = sheet3)
 
# concatenate dfs vertically
df = pd.concat([df1, df2, df3], axis=0)# 
print(df.shape)
print(df1.shape[0]+df2.shape[0]+ df3.shape[0])

##convert week to date 
df['date'] = pd.to_datetime(df['week'])
df = df.sort_values(by = 'date')
df = df.reset_index(drop=True)
print(df.dtypes)
df

#get year month
df['yearMonth'] = df['date'].dt.to_period('M')
print(df.dtypes)
df

##add column to count rows
df['observed'] = 1
df

df_grp = df.groupby(['state', 'yearMonth'], as_index=False).sum(numeric_only=True)
print(df_grp.dtypes)
df_grp
count_state_rows = df_grp.groupby('state').size()
count_state_rows ## 
 

# get all possible combination of state and yearMonth 
dates = list(pd.date_range(start='1/1/2008', end='12/1/2022', freq='MS')) ## 
states = list(df_grp.state.unique())
all_combinations = list(itertools.product(states, dates))
df_allComb = pd.DataFrame(all_combinations, columns=['state','yearMonth']) ## 
df_allComb['yearMonth'] = df_allComb['yearMonth'].dt.to_period('M')

## merge df_allComb and df_grp
df_allComb = df_allComb.merge(df_grp,how='left' ).fillna(0)  
print(df_allComb.dtypes)
print(df_allComb.isnull().values.any()) ## 
df_allComb ## 

## convert state names to uppercase
df_allComb['state'] = df_allComb['state'].str.upper()
print(df_allComb['state'].unique())

# inset year, month  columns (
df_allComb.insert(2,'year', df_allComb['yearMonth'].dt.strftime('%Y'))  #
df_allComb['year'] = df_allComb['year'].astype('int64')
df_allComb.insert(3,'month', df_allComb['yearMonth'].dt.strftime('%m')) ##
df_allComb['month'] = df_allComb['month'].astype('int64')
 

df_allComb


## Format population data

- population data from https://www.abs.gov.au/statistics/people/population/national-state-and-territory-population/latest-release#data-downloads-data-cubes : file used is: population -states and territories
- this file contains quarterly ERP for all states from june 1981 to dec 2023: Males and Females populations are in separate colmns 



In [ ]:
##read population data
df_pop = pd.read_csv(path +  'data/population/abs_ERP_sub.csv', skipinitialspace = True,  low_memory=False) 
df_pop #

df_pop_long=pd.melt(df_pop, id_vars=['date'],
               var_name = 'state', value_name = 'qrtlyPopn') 
print(df_pop_long['state'].unique()) #[
df_pop_long

#format month
df_pop_long.insert(1,'monthName', df_pop_long['date'].str[:3]) #
month_mapping = {
             'Mar' : 3,'Jun': 6, 'Sep': 9 , 'Dec': 12 
}
df_pop_long
### format year
df_pop_long.insert(3,'year', df_pop_long['date'].str[-2:].astype('int')) #get year

df_pop_long #
df_pop_long_sub = df_pop_long.copy()
df_pop_long_sub = df_pop_long_sub[df_pop_long_sub['year'].between(8, 23)] #
print(df_pop_long_sub['year'].unique()) #
df_pop_long_sub['year'] = df_pop_long_sub['year']+2000
df_pop_long_sub

df_pop_long_sub.insert(2,'month', df_pop_long_sub['monthName'].apply(str).map(month_mapping).astype(int)) 
df_pop_long_sub

#get yearMonth from year newMonth columns
df_pop_long_sub.insert(4,'yearMonth', pd.to_datetime(df_pop_long_sub[['year', 'month']].assign(DAY=1)))
df_pop_long_sub['yearMonth'] = df_pop_long_sub['yearMonth'].dt.to_period('M')


df_pop_long_sub  ##

## Merge influenza and population datasets

In [ ]:
df_merged = df_allComb.merge(df_pop_long_sub,how='left' )
df_merged['popn'] = df_merged['qrtlyPopn'].bfill(limit=2) ## 

# remove unwanted columns 
df_merged = df_merged.drop(['date', 'monthName'], axis=1)

## save data
# df_merged.to_csv(path + 'data/pathogen/influenza_by_state_by_yearMonth_18Sept2024.csv', index = False)


print(df_merged.dtypes)
df_merged

## Merge health data with climate data

In [ ]:
##read influenza data upto Dec 2019
df_flu = pd.read_csv(path + 'data/pathogen/influenza_by_state_by_yearMonth_18Sept2024.csv', low_memory=False) ## 
df_flu['date'] = pd.to_datetime(df_flu[['year', 'month']].assign(DAY=1)) 
df_flu = df_flu[['state', 'date', 'year', 'month', 'observed', 'popn', 'season']]  
# print(df_flu.state.unique()) ##  
# print(df_flu.year.unique()) ## 
# print(df_flu.dtypes)

### load cleaned barra2 climate data 
df_barra = pd.read_csv(path + 'data/climate_data/barra2/barra2_allAU.csv', low_memory=False) 
df_barra['date'] = pd.to_datetime(df_barra[['year', 'month']].assign(DAY=1))
df_barra = df_barra[df_barra['year'] > 2007]  
# print(df_barra.jurisdiction.unique())  ##
# print(df_barra.year.unique()) ###

df_barra = df_barra[['jurisdiction', 'date', 'year', 'month', 'tas',  'RH']] ## 

df_barra=df_barra.rename(columns = {'jurisdiction':'state'}) ## 

df_barra

### save combined data
df_combined.to_csv(path + '/data/pathogen/combinedInflClimate.csv', index = False)

# Impute missing weekly data (After reviewers comments)

In [ ]:
#load raw data and sort by 'week' column
sheet1 = '2008-2015' #  
sheet2 = '2016-2018' 
sheet3 = '2019-2022' 
file_name = path + '/data/pathogen/nndss_influenza_raw_data.xlsx' #  
df1 = read_excel(file_name, sheet_name = sheet1)# 
df2 = read_excel(file_name, sheet_name = sheet2)# 
df3 = read_excel(file_name, sheet_name = sheet3)# 
 
# concatenate dfs vertically
df = pd.concat([df1, df2, df3], axis=0)# 
print(df.shape)
print(df1.shape[0]+df2.shape[0]+ df3.shape[0])

##convert week to date 
df['date'] = pd.to_datetime(df['week'])
df = df.sort_values(by = 'date')
df = df.reset_index(drop=True)
print(df.dtypes)
df

##add column to count rows
df['observed'] = 1
df


### group by date and state
df_grp = df.groupby(['state', 'date'], as_index=False).sum(numeric_only=True)
print(df_grp.dtypes)
df_grp ## 5053 rows

## create a dataframe with all combinations of state, date
dates = pd.date_range(start='2008-01-04',  end='2022-12-30',  freq='W-FRI')
states = df_grp['state'].unique()
all_combinations = itertools.product(states, dates)
all_combinations
df_allComb = pd.DataFrame(all_combinations,  columns=['state', 'date']) ##  
iso = df_allComb['date'].dt.isocalendar()
df_allComb


## merge dataframes
df_allComb = df_allComb.merge(df_grp,how='left' )## 
print(df_allComb.dtypes)
print(df_allComb.isnull().values.any()) ##  
df_allComb ##  

df_allComb['year'] = iso.year
df_allComb['week'] = iso.week
df_allComb['month'] = df_allComb['date'].dt.month
df_allComb

### filter data excluding covid period and find which dates has missing values
df_excld_covid = df_allComb[~df_allComb['year'].isin([2020, 2021])] ##  
print(df_allComb.year.unique())
df_excld_covid  ## 

### count missing numbers for each state
df_missing = df_excld_covid[df_allComb['observed'].isna()]

missing_count = (
    df_missing.groupby('state')
    .size()
    .reset_index(name='n_missing')
)

missing_count ##  

### get missing weeks 
missing_weeks = (
    df_missing[['state', 'date', 'year', 'month', 'week']]
    .sort_values(['state', 'date'])
)


missing_weeks

### Impute weekly data 

In [ ]:

# ### define a function for imputation

def impute_adjacent_weeks_by_date(group):

    group = group.copy()

    # Ensure dates are datetime
    group['date'] = pd.to_datetime(group['date'])

    # Start with observed values
    group['observed_imp'] = group['observed']

    # Missing observations only
    missing_idx = group[group['observed'].isna()].index

    for idx in missing_idx:

        target_date = group.loc[idx, 'date']

        imputed = False

        # progressively widen the search window
        for window in [2, 3, 5]:

            start_date = target_date - pd.Timedelta(weeks=window)
            end_date   = target_date + pd.Timedelta(weeks=window)

            vals = group.loc[
                (group['date'] >= start_date) &
                (group['date'] <= end_date) &
                (group.index != idx),
                'observed'
            ].dropna()

            if len(vals) > 0:

                group.loc[idx, 'observed_imp'] = vals.median()

                imputed = True
                break

        if not imputed:
            group.loc[idx, 'observed_imp'] = np.nan

    return group


####################################




In [ ]:
df_impute = (
    df_excld_covid.copy()
    .sort_values(['state', 'year', 'week'])
    .groupby('state') ## 
    .apply(impute_adjacent_weeks_by_date)
    .reset_index(level = 0)
    .reset_index(drop=True)
) ## 

## if there remains a row with unimputed observations, then assume 0 cases  
df_impute['observed_imp'] = (
    df_impute['observed_imp']
    .fillna(0)
)

# ### Crate a column for filling observations with 0 (initial modelling approach) and a column indicating missing week
df_impute['observed_filled'] = (df_impute['observed'].fillna(0))
df_impute['missing_week'] = df_impute['observed'].isna()


#######################################################

df_impute

In [6]:
### Check largest weekly imputations 
# df_check = df_impute.loc[
#     df_impute['observed'].isna(),
#     ['state','date','week','year','observed_imp']
# ]

# df_check.sort_values(
#     'observed_imp',
#     ascending=False
# ).head(30)



In [ ]:
####### Compare the two imputation methods 

total_original = df_impute['observed_filled'].sum()
total_imputed = df_impute['observed_imp'].sum()

print(total_original) ## 1280517.0
print(total_imputed) ## 1280867.5

print( 100 * (total_imputed - total_original)  / total_original) ## 0.027%

## compare by state
df_compare = (df_impute
    .groupby('state')
    .agg(total_original = ('observed_filled','sum'),
        total_imputed=('observed_imp','sum')
    )
    .reset_index(level = 0)
    .reset_index(drop=True)
)

## compute absolute difference 
df_compare['abs__pct_diff'] = 100*((df_compare['total_imputed']- df_compare['total_original']).abs())/df_compare['total_original']
df_compare

### Create a dataframe for monthly scale 

In [ ]:
df_impute_monthly = df_impute.copy()

## aggregate to state-year-month
df_impute_monthly = (df_impute_monthly
    .groupby(['state', 'year', 'month'], as_index=False)
    .agg(observed=('observed', 'sum'),
        observed_filled=('observed_filled', 'sum'),
        observed_imp=('observed_imp', 'sum'),
        missing_week = ('missing_week', 'sum')
    )
)

## check columns for any missing values on monthly scale
print(f"Missing number in original observations: {df_impute_monthly['observed'].isna().sum()}")
print(f"Missing number in the local impputation: {df_impute_monthly['observed_imp'].isna().sum()}")
print(f"Missing number after assuming 0 cases: {df_impute_monthly['observed_filled'].isna().sum()}")

## create a proper year-month date
df_impute_monthly['date'] = pd.to_datetime(
    df_impute_monthly['year'].astype(str) + '-' + df_impute_monthly['month'].astype(str)  + '-01'
)

print(df_impute_monthly.dtypes)

# ## convert state names to uppercase
df_impute_monthly['state'] = df_impute_monthly['state'].str.upper()
print(df_impute_monthly['state'].unique())

####################################################
### save monthly imputed data 
# df_impute_monthly.to_csv(path + '/data/pathogen/imputed_data_monthly_scale.csv', index = False)
#######################################################

df_impute_monthly



In [ ]:
## summarise the impact of imputation

## compute absolute difference 
df_compare_monthly = df_impute_monthly.copy()

df_compare_monthly['abs_diff'] = (df_compare_monthly['observed_imp'] - df_compare_monthly['observed_filled']).abs()

df_compare_monthly

state_summary = (df_compare_monthly
    .groupby('state')
    .agg(
        mean_diff=('abs_diff', 'mean'),
        max_diff=('abs_diff', 'max'),
        n_months=('abs_diff', 'count'),
        n_missing_week = ('missing_week', 'sum')
    )
    .reset_index()
)

state_summary

 

In [292]:
# ### check whichstate,  months contribute to largest differences
# df_impute_monthly.sort_values(
#     'abs_diff',
#     ascending=False
# ).head(20)


## Combine the imputed dataframe with climate and population data

In [ ]:
### read the combined climate and influenza data that was previously used for modelling 

df_prev =  pd.read_csv(path + '/data/pathogen/combinedInflClimate.csv', low_memory=False) ## 

df_prev = df_prev[['state', 'date', 'year', 'month', 'observed', 'popn', 'tas', 'RH']] ##  
df_prev['date'] = pd.to_datetime(df_prev['date']) #  
df_prev=df_prev[~df_prev['year'].isin([2020, 2021])].reset_index(drop=True) ##  

df_prev = df_prev.sort_values(by=['state', 'date'], ignore_index=True)


print(df_prev.year.unique())
print(df_prev.dtypes)
df_prev

#################
## read the imputed influenza monthly data

df_flu =  pd.read_csv(path + '/data/pathogen/imputed_data_monthly_scale.csv', low_memory=False) ##  
df_flu['date'] = pd.to_datetime(df_flu['date']) #  
df_flu = df_flu[['state', 'date', 'year', 'month', 'observed', 'observed_imp']]

df_flu = df_flu.sort_values(by=['state', 'date'], ignore_index=True)
print(df_flu.dtypes)
df_flu

####################
## combine the two datasets
df_combined = (df_flu[['state', 'date', 'year', 'month', 'observed', 'observed_imp'] ]
    .merge(
        df_prev[['state', 'date', 'popn', 'tas', 'RH']],
        on=['state', 'date'],
        how='left'
    )
)
df_combined

#######################################
### check for missing values introduced during merge

print(df_combined[['popn', 'tas', 'RH']].isna().sum())
####################################################
### save combined data
# df_combined.to_csv(path + '/data/pathogen/imputed_combinedInflClimate.csv', index = False)
#######################################################